In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
import sys

In [2]:
# 参考リンク : https://dev.sy0osetu.com/man/api/ 

# APIのエンドポイント
NAROU_API_URL = "https://api.syosetu.com/novelapi/api/"
# 1リクエストあたりの待機時間（秒）
REQUEST_INTERVAL = 1

In [3]:
def get_narou_data(start_index=1, limit=500, order="hyoka", genre=None):
    """
    なろう小説APIから指定された条件で小説データを取得
    """
    params = {
        "out": "json",
        "lim": limit,
        "st": start_index,
        "order": order,
        "of": "n-t-s-w-k-gl-l-nt-e-a-f-gf-ga-ti-dp-wp-mp-qp-yp-imp-r-ah-ka",
    }
    
    if genre:
        params["genre"] = genre

    try:
        response = requests.get(NAROU_API_URL, params=params)
        response.raise_for_status()
        data = response.json()
        
        if isinstance(data, list) and len(data) > 1 and isinstance(data[1], dict):
            return data[1:]
        else:
            return []
            
    except requests.exceptions.RequestException as e:
        print(f"APIリクエスト中にエラーが発生しました: {e}")
    except ValueError as e:
        print(f"API応答のJSON解析に失敗しました: {e}")
        
    return None

In [ ]:
def create_corpus(order_list, num_per_order=2000, output_filename="narou_corpus_with_genre.csv", genre=None, genre_name="all"):
    """
    複数の並び順で小説データを収集し、重複を削除して保存します。
    """
    combined_novel_data = pd.DataFrame()
    limit = 500
    
    print(f"【ジャンル: {genre_name}】のデータ収集を開始します。各並び順で最大{num_per_order}件取得します。")

    for order in order_list:
        all_novel_data = []
        max_requests = (num_per_order // limit)

        print(f"\n--- {order}順でのデータ収集 ({num_per_order}件) ---")
        for i in range(max_requests):
            start = i * limit + 1
            
            print(f"{i+1}回目のリクエスト (通算{start}件目～)...")
            novels = get_narou_data(start_index=start, limit=limit, order=order, genre=genre)

            if novels is None:
                print("APIリクエストに失敗したため、この並び順での処理を中断します。")
                break
            
            if not novels:
                print("これ以上、有効な小説データを取得できませんでした。")
                break

            all_novel_data.extend(novels)
            time.sleep(REQUEST_INTERVAL)

        if all_novel_data:
            df_temp = pd.DataFrame(all_novel_data)
            
            if genre:
                df_temp['genre'] = genre

            combined_novel_data = pd.concat([combined_novel_data, df_temp], ignore_index=True)
            print(f"{order}順で{len(df_temp)}件のデータを取得しました。")
            print(f"現在の合計件数（重複あり）: {len(combined_novel_data)}件")
            
    if combined_novel_data.empty:
        print(f"\n【ジャンル: {genre_name}】では小説データを1件も取得できませんでした。")
        return

    # 重複の削除
    print(f"\n合計 {len(combined_novel_data)}件 のデータから重複を削除します...")
    df = combined_novel_data.drop_duplicates(subset=['ncode'], keep='first').copy()
    print(f"重複削除後、{len(df)}件のユニークな小説データになりました。")

    # 短編小説（noveltypeが2）を除外する
    if 'noveltype' in df.columns:
        print(f"\n短編小説を除外します...")
        num_before_filter = len(df)
        df = df[df['noveltype'] != 2].copy()
        num_after_filter = len(df)
        print(f"短編小説 {num_before_filter - num_after_filter}件 を除外し、{num_after_filter}件の連載作品が残りました。")
    else:
        print("警告: 'noveltype'列が見つからないため、短編小説の除外をスキップしました。")

    # 列名を日本語にリネーム
    df.rename(columns={
        'ncode': 'Nコード', 'title': 'タイトル', 'story': 'あらすじ',
        'writer': '作者', 'keyword': 'キーワード', 'general_lastup': '最終更新日',
        'general_firstup': '初回投稿日', 'general_all_no': '投稿話数',
        'length': '文字数', 'noveltype': '小説タイプ', 'end': '完結フラグ',
        'all_point': '総合評価ポイント', 'fav_novel_cnt': 'ブックマーク数',
        'genre': '作品ジャンル',
        'time': '読了時間',
        'daily_point': '日間ポイント',
        'weekly_point': '週間ポイント',
        'monthly_point': '月間ポイント',
        'quarter_point': '四半期ポイント',
        'yearly_point': '年間ポイント',
        'impression_cnt': '感想数',
        'review_cnt': 'レビュー数',
        'all_hyoka_cnt': '評価者数',
        'kaiwaritsu': '会話率'
    }, inplace=True)
    
    # 「小説タイプ」の値を分かりやすい文字列に変換
    if '小説タイプ' in df.columns:
        noveltype_map = {1: '連載'}
        df['小説タイプ'] = df['小説タイプ'].map(noveltype_map)
    
    # 必須列の存在チェック
    required_columns = ['最終更新日', '初回投稿日', '投稿話数']
    for col in required_columns:
        if col not in df.columns:
            print(f"\nエラー: '{col}' のデータが取得できませんでした。")
            sys.exit()

    # データ型の変換と計算
    df['完結フラグ'] = pd.to_numeric(df['完結フラグ'], errors='coerce').fillna(0).astype(int)
    df['最終更新日_dt'] = pd.to_datetime(df['最終更新日'])
    df['初回投稿日_dt'] = pd.to_datetime(df['初回投稿日'])

    df['投稿期間_日数'] = (df['最終更新日_dt'] - df['初回投稿日_dt']).dt.days
    df['更新頻度_日数/話'] = df.apply(
        lambda row: row['投稿期間_日数'] / (row['投稿話数'] - 1) if row['投稿話数'] > 1 else (0 if row['投稿話数'] == 1 and row['投稿期間_日数'] == 0 else None),
        axis=1
    )

    one_year_ago = datetime.now() - timedelta(days=365)
    is_eternal_condition = (df['完結フラグ'] == 0) & (df['最終更新日_dt'] < one_year_ago)
    df['is_eternal'] = is_eternal_condition.astype(int)
    
    print(f"\nフィルタリング前のデータ件数: {len(df)}件")
    
    # アクティブな連載作品（未完結かつ1年以内に更新あり）を除外
    condition_to_exclude = (df['完結フラグ'] == 0) & (df['is_eternal'] == 0)
    
    num_before_filter = len(df)
    df = df[~condition_to_exclude].copy()
    num_after_filter = len(df)
    
    print(f"フィルタリングにより、アクティブな連載作品 {num_before_filter - num_after_filter}件 を除外しました。")
    print(f"フィルタリング後のデータ件数: {num_after_filter}件")

    # 不要な日時列を削除
    df = df.drop(columns=['最終更新日_dt', '初回投稿日_dt', '投稿期間_日数'])

    # CSVファイルとして保存
    df.to_csv("corpus/sub/"+output_filename, index=False, encoding='utf-8-sig')
    print(f"\nデータ収集が完了し、'{output_filename}' に保存しました。")

In [5]:
# 取得したいジャンルを定義（キー：ファイル名用、値：API用のジャンル番号）
genres_to_fetch = {
    '未選択': 0,
    '異世界(恋愛)': 101,
    '現実世界(恋愛)': 102,
    'ハイファンタジー': 201,
    'ローファンタジー': 202,
    '純文学': 301,
    'ヒューマンドラマ': 302,
    '歴史': 303,
    '推理': 304,
    'ホラー': 305,
    'アクション': 306,
    'コメディー': 307,
    'VRゲーム': 401,
    '宇宙': 402,
    '空想科学': 403,
    'パニック': 404,
    '童話': 9901,
    '詩': 9902,
    'エッセイ': 9903,
    'リプレイ': 9904,
    'その他': 9999,
    'ノンジャンル': 9801
}

# 各並び順で取得する小説の件数（最大2000件）
NUM_OF_NOVELS_TO_FETCH = 2000
order_to_fetch = ["new","favnovelcnt","reviewcnt","hyoka","hyokaasc","dailypoint","weeklypoint","monthlypoint","quarterpoint","yearlypoint","impressioncnt","hyokacnt","hyokacntasc","weekly","lengthdesc","lengthasc","generalfirstup","ncodeasc","ncodedesc","old"]
order_to_fetch_lower = ["hyokaasc","hyokacntasc","lengthasc"]

# 定義したジャンルごとにループ処理を実行
for name, genre_code in genres_to_fetch.items():
    output_file = f"narou_corpus_{name}.csv"
    create_corpus(
        order_to_fetch, 
        num_per_order=NUM_OF_NOVELS_TO_FETCH,
        output_filename=output_file,
        genre=genre_code,
        genre_name=name
    )
    print(f"\n{'='*50}\n")

print("すべてのジャンルのデータ収集が完了しました。")

【ジャンル: 未選択】のデータ収集を開始します。各並び順で最大2000件取得します。

--- new順でのデータ収集 (2000件) ---
1回目のリクエスト (通算1件目～)...
2回目のリクエスト (通算501件目～)...
3回目のリクエスト (通算1001件目～)...
4回目のリクエスト (通算1501件目～)...
new順で2000件のデータを取得しました。
現在の合計件数（重複あり）: 2000件

--- favnovelcnt順でのデータ収集 (2000件) ---
1回目のリクエスト (通算1件目～)...
2回目のリクエスト (通算501件目～)...
3回目のリクエスト (通算1001件目～)...
4回目のリクエスト (通算1501件目～)...
favnovelcnt順で2000件のデータを取得しました。
現在の合計件数（重複あり）: 4000件

--- reviewcnt順でのデータ収集 (2000件) ---
1回目のリクエスト (通算1件目～)...
2回目のリクエスト (通算501件目～)...
3回目のリクエスト (通算1001件目～)...
4回目のリクエスト (通算1501件目～)...
reviewcnt順で2000件のデータを取得しました。
現在の合計件数（重複あり）: 6000件

--- hyoka順でのデータ収集 (2000件) ---
1回目のリクエスト (通算1件目～)...
2回目のリクエスト (通算501件目～)...
3回目のリクエスト (通算1001件目～)...
4回目のリクエスト (通算1501件目～)...
hyoka順で2000件のデータを取得しました。
現在の合計件数（重複あり）: 8000件

--- hyokaasc順でのデータ収集 (2000件) ---
1回目のリクエスト (通算1件目～)...
2回目のリクエスト (通算501件目～)...
3回目のリクエスト (通算1001件目～)...
4回目のリクエスト (通算1501件目～)...
hyokaasc順で2000件のデータを取得しました。
現在の合計件数（重複あり）: 10000件

--- dailypoint順でのデータ収集 (2000件) ---
1回目のリクエスト (通算1件目～)...
2回目のリクエスト (

In [6]:
import glob
# corpusフォルダ内のCSVファイルをすべて読み込む
path = "corpus/sub/"
all_files = glob.glob(path + "narou_corpus_*.csv")

li = []
for filename in all_files:
    df = pd.read_csv(filename, index_col=None, header=0)
    li.append(df)

filename = "narou_corpus.csv"

if not li:
    print("統合するCSVファイルが見つかりませんでした。ファイル名やパスを確認してください。")
else:
    # すべてのデータフレームを結合
    combined_df = pd.concat(li, axis=0, ignore_index=True)
    print(f"結合前の合計件数: {len(combined_df)}件")

    # ジャンル間で重複している作品を削除
    final_df = combined_df.drop_duplicates(subset=['Nコード'], keep='first')
    print(f"重複削除後の最終的な件数: {len(final_df)}件")

    # 最終的なコーパスを保存
    final_df.to_csv("corpus/"+filename, index=False, encoding='utf-8-sig')
    print(f"最終的なコーパス '{filename}' を保存しました。")

    # 最終的なis_eternalラベルの件数を表示
    if 'is_eternal' in final_df.columns:
        print("\n--- 最終コーパスのデータ内訳 ---")
        print("is_eternalラベルの件数:")
        print(final_df['is_eternal'].value_counts())
        print("---------------------------------")
    else:
        print("\n'is_eternal'列がデータ内に見つかりませんでした。")

結合前の合計件数: 127405件
重複削除後の最終的な件数: 116696件
最終的なコーパス 'narou_corpus.csv' を保存しました。

--- 最終コーパスのデータ内訳 ---
is_eternalラベルの件数:
is_eternal
0    75899
1    40797
Name: count, dtype: int64
---------------------------------
